<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">

# **Operaciones de Aprendizaje Automático III**
# **Clase 6, Destilación de conocimiento con DoRA**


Un modelo maestro grande (Qwen3-1.7B) le enseña a un alumno chico (Qwen3-0.6B)
a responder consultas sobre un reglamento de posgrado usando solo el contexto dado.

El alumno no copia solo el texto del maestro: aprende su distribución de
probabilidad en cada token. Si el maestro dudaba entre "treinta" y "30", el alumno
aprende las dos opciones y cuánto pesaba cada una.

Pasos:

1. Datos: preguntas con su contexto y una respuesta de referencia
2. El maestro responde y se guardan sus 32 candidatos más probables en cada token
3. Se descartan las respuestas malas
4. El alumno se entrena con DoRA para imitar al maestro
5. Se compara el alumno antes y después

In [ ]:
!pip install -q -U transformers peft accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 28.1 MB/s eta 0:00:00


In [ ]:
import gc
import importlib
import math
import random
import re
import unicodedata

import numpy as np
import torch
import torch.nn.functional as F
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

# colab trae una version vieja de torchao y peft falla al revisarla con un error de
# version incompatible, se le indica a peft que torchao no esta
for nombre in ["peft.import_utils", "peft.tuners.lora.torchao", "peft.utils.quantization_utils"]:
    try:
        modulo = importlib.import_module(nombre)
        modulo.is_torchao_available = lambda: False
    except ImportError:
        pass

# los dos modelos son de la familia qwen3 y comparten tokenizador, condicion necesaria para comparar sus distribuciones token por token
modelo_maestro = "Qwen/Qwen3-1.7B"
modelo_alumno = "Qwen/Qwen3-0.6B"

# generacion del maestro
top_k = 32 # candidatos que se guardan por token
max_tokens = 64  # las respuestas son de una o dos oraciones
lote_generacion = 16

# entrenamiento del alumno
usar_dora = True  # con false se entrena con lora comun
rango = 16

epocas = 3
lote = 4
tasa_aprendizaje = 2e-4
peso_kl = 0.7  # cuanto pesa imitar la distribucion del maestro
peso_ce = 0.3  # cuanto pesa el token que el maestro eligio
temperatura = 2.0  # suaviza las distribuciones antes de compararlas

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
# bfloat16 necesita una gpu ampere o mas nueva; la t4 usa float16
if dispositivo == "cuda" and torch.cuda.get_device_capability()[0] >= 8:
    tipo = torch.bfloat16
elif dispositivo == "cuda":
    tipo = torch.float16
else:
    tipo = torch.float32
print("dispositivo:", dispositivo, tipo)

dispositivo: cuda torch.float16


**1. Datos**

* diez artículos de un reglamento, cada uno tiene un contexto
* cinco formas de preguntar lo mismo y una respuesta de referencia
* cada consulta recibe el documento correcto y otro de un tema distinto
* las tres primeras preguntas de cada tema son para entrenar
y las dos últimas para evaluar: así se mide si el alumno responde preguntas que
nunca vio

In [ ]:
reglamento = [
    ("La inscripcion a las materias del posgrado se habilita dos semanas antes del inicio del cuatrimestre y cierra el viernes de la primera semana de cursada.",
     ["¿Hasta cuando puedo inscribirme a una materia?", "¿Cual es la fecha limite de inscripcion?",
      "¿Cuando abre la inscripcion a materias?", "Me inscribo tarde, ¿todavia llego?",
      "Perdi el plazo de inscripcion, ¿que hago?"],
     "Se habilita dos semanas antes del cuatrimestre y cierra el viernes de la primera semana de cursada."),
    ("Se pierde la regularidad de una materia con mas de dos ausencias injustificadas o con una nota menor a cuatro en el trabajo final.",
     ["¿Cuando se pierde la regularidad?", "¿Cuantas faltas puedo tener?",
      "¿Que nota minima necesito en el trabajo final?", "Falte tres veces, ¿quedo libre?",
      "¿Como se mantiene la condicion de regular?"],
     "Con mas de dos ausencias injustificadas o una nota menor a cuatro en el trabajo final."),
    ("Las equivalencias se solicitan por nota dirigida a la direccion del posgrado, adjuntando el programa de la materia cursada y el certificado analitico.",
     ["¿Que necesito para pedir una equivalencia?", "¿Como presento una equivalencia?",
      "¿A quien dirijo el pedido de equivalencia?", "Ya curse algo parecido en otra facultad, ¿sirve?",
      "¿Que documentacion piden para equivalencias?"],
     "Una nota a la direccion del posgrado con el programa de la materia y el certificado analitico."),
    ("La defensa del trabajo final se solicita con treinta dias de anticipacion y requiere el aval escrito del director.",
     ["¿Con cuanta anticipacion se pide la defensa?", "¿Como pido fecha de defensa?",
      "¿Necesito el aval del director para defender?", "Quiero defender el mes que viene, ¿llego?",
      "¿Que tramite previo tiene la defensa?"],
     "Con treinta dias de anticipacion y el aval escrito del director."),
    ("El arancel se abona en diez cuotas mensuales. La mora superior a sesenta dias suspende la inscripcion a nuevas materias.",
     ["¿Que pasa si me atraso con el pago?", "¿En cuantas cuotas se paga?",
      "Estoy atrasado con las cuotas, ¿me bloquean?", "¿Que pasa si dejo de pagar dos meses?",
      "¿Como es el plan de pago del posgrado?"],
     "Se paga en diez cuotas; una mora mayor a sesenta dias suspende la inscripcion a nuevas materias."),
    ("El director de trabajo final debe tener titulo de posgrado y se designa por resolucion de la comision academica.",
     ["¿Quien puede ser director de trabajo final?", "¿Que requisitos tiene un director?",
      "¿Quien aprueba la designacion del director?", "¿Puede dirigirme alguien sin posgrado?",
      "Quiero cambiar de director, ¿que necesita tener el nuevo?"],
     "Alguien con titulo de posgrado, designado por resolucion de la comision academica."),
    ("La prorroga para entregar el trabajo final es de seis meses y se otorga una sola vez, con informe de avance del director.",
     ["¿Cuanto dura la prorroga del trabajo final?", "¿Puedo pedir mas tiempo para el trabajo final?",
      "¿Cuantas veces se puede pedir prorroga?", "¿Que piden para otorgar la prorroga?",
      "Se me vence el plazo del trabajo final, ¿que opciones tengo?"],
     "Seis meses, por unica vez y con informe de avance del director."),
    ("La cursada remota requiere camara encendida durante las evaluaciones y conexion propia; la institucion no provee equipamiento.",
     ["¿Que condiciones tiene la cursada remota?", "¿Tengo que prender la camara?",
      "¿Que requisitos tecnicos tiene cursar remoto?", "¿La facultad presta computadora para cursar a distancia?",
      "¿Como son las evaluaciones en la modalidad remota?"],
     "Camara encendida en las evaluaciones y conexion propia; la institucion no provee equipamiento."),
    ("El tramite de titulo se inicia una vez aprobadas todas las materias y la defensa, y demora entre cuatro y seis meses.",
     ["¿Cuanto demora el titulo?", "¿Cuanto tarda en salir el diploma?",
      "¿Cuando puedo empezar el tramite de titulo?", "Ya defendi, ¿cuanto espero el titulo?",
      "¿Que falta para iniciar el tramite del diploma?"],
     "Se inicia con todas las materias y la defensa aprobadas, y demora entre cuatro y seis meses."),
    ("La baja de una materia sin penalidad se puede pedir hasta la tercera clase; despues figura como ausente en el acta.",
     ["¿Hasta cuando puedo dar de baja una materia?", "¿Hasta que clase puedo darme de baja?",
      "¿La baja tiene penalidad?", "¿Puedo abandonar una materia sin que figure?",
      "Deje de ir en la quinta clase, ¿que queda en el acta?"],
     "Hasta la tercera clase sin penalidad; despues figura como ausente en el acta."),
]

# distintas formas en que la gente escribe la misma consulta
variantes = ["{}", "hola, {}", "consulta: {}", "buenas tardes. {}"]

entrenamiento = []
evaluacion = []
for i, (contexto, preguntas, referencia) in enumerate(reglamento):
    otros = [r[0] for j, r in enumerate(reglamento) if j != i]
    for k, pregunta in enumerate(preguntas):
        documentos = [contexto, random.choice(otros)]
        random.shuffle(documentos)
        item = {"contexto": "\n\n".join(documentos), "pregunta": pregunta, "referencia": referencia}
        if k < 3:
            for v in variantes:
                entrenamiento.append(dict(item, pregunta=v.format(pregunta)))
        else:
            evaluacion.append(item)

print("ejemplos de entrenamiento:", len(entrenamiento))
print("preguntas de evaluacion:", len(evaluacion))
print(entrenamiento[1])

ejemplos de entrenamiento: 120
preguntas de evaluacion: 20
{'contexto': 'La inscripcion a las materias del posgrado se habilita dos semanas antes del inicio del cuatrimestre y cierra el viernes de la primera semana de cursada.\n\nLa cursada remota requiere camara encendida durante las evaluaciones y conexion propia; la institucion no provee equipamiento.', 'pregunta': 'hola, ¿Hasta cuando puedo inscribirme a una materia?', 'referencia': 'Se habilita dos semanas antes del cuatrimestre y cierra el viernes de la primera semana de cursada.'}


**2. El maestro responde y se guardan sus distribuciones**

* maestro y alumno usan la misma instrucción y la misma plantilla, con
enable_thinking=False, Qwen3 responde directo, sin escribir antes su razonamiento.

* la clave es output_logits=True: además del texto, el maestro devuelve sus
probabilidades en cada paso
* de cada paso se guardan los 32 tokens más probables
* guardar el vocabulario completo, de 151.936 tokens, ocuparía demasiado.

In [ ]:
instruccion = (
    "Respondes consultas sobre el reglamento usando unicamente el contexto dado. "
    "Si el contexto no alcanza, dilo. Respuesta breve, una o dos oraciones."
)

tokenizador = AutoTokenizer.from_pretrained(modelo_maestro)
tokenizador.padding_side = "left"   # para generar en lote el relleno va a la izquierda
if tokenizador.pad_token is None:
    tokenizador.pad_token = tokenizador.eos_token


def construir_prompt(item):
    mensajes = [
        {"role": "system", "content": instruccion},
        {"role": "user", "content": "Contexto:\n" + item["contexto"] + "\n\nConsulta: " + item["pregunta"]},
    ]
    return tokenizador.apply_chat_template(mensajes,
                                           tokenize=False,
                                           add_generation_prompt=True,
                                           enable_thinking=False)


maestro = AutoModelForCausalLM.from_pretrained(modelo_maestro, dtype=tipo).to(dispositivo)
maestro.eval()

# tokens que marcan el fin de la respuesta
fin = maestro.generation_config.eos_token_id
# le preguntamos al modelo qué token o tokens usa para terminar de generar
# según el modelo, la respuesta puede ser un número solo (151645) o una lista ([151645, 151643])
ids_fin = set(fin if isinstance(fin, list) else [fin]) | {tokenizador.eos_token_id}

generaciones = []
for i in range(0, len(entrenamiento), lote_generacion):
    items = entrenamiento[i:i + lote_generacion]
    entrada = tokenizador([construir_prompt(x) for x in items], return_tensors="pt",
                          padding=True, add_special_tokens=False).to(dispositivo)
    with torch.no_grad():
        salida = maestro.generate(
            **entrada,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            pad_token_id=tokenizador.pad_token_id,
            return_dict_in_generate=True,
            output_logits=True,   # logits crudos del maestro en cada paso
        )
    largo_prompt = entrada["input_ids"].shape[1]
    nuevos = salida.sequences[:, largo_prompt:]
    logits = torch.stack(salida.logits, dim=1).float()   # (lote, pasos, vocabulario)

    for f, item in enumerate(items):
        ids = nuevos[f].tolist()
        largo = len(ids)
        termino = False
        for t, token in enumerate(ids):
            if token in ids_fin:
                largo = t + 1
                termino = True
                break
        probs, candidatos = F.softmax(logits[f, :largo], dim=-1).topk(top_k, dim=-1)
        generaciones.append({
            "prompt": entrada["input_ids"][f][entrada["attention_mask"][f].bool()].tolist(),
            "respuesta": ids[:largo],
            "texto": tokenizador.decode(ids[:largo], skip_special_tokens=True).strip(),
            "termino": termino,
            "candidatos": candidatos.cpu(),   # (largo, top_k) ids de los tokens
            "probs": probs.cpu(),             # (largo, top_k) sus probabilidades
        })
    print(len(generaciones), "de", len(entrenamiento))

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

16 de 120
32 de 120
48 de 120
64 de 120
80 de 120
96 de 120
112 de 120
120 de 120


Así se ve lo que se guardó: para cada token de una respuesta, cuál eligió el maestro
y qué otros candidatos consideraba.

In [ ]:
g = generaciones[0]
print("respuesta del maestro:", g["texto"], "\n")
for t in range(min(6, len(g["respuesta"]))):
    elegido = tokenizador.decode([g["respuesta"][t]])
    otros = [(tokenizador.decode([int(c)]), round(float(p), 2)) for c, p in zip(g["candidatos"][t][:4], g["probs"][t][:4])]
    print(repr(elegido), "->", otros)

respuesta del maestro: Se puede inscribir dos semanas antes del inicio del cuatrimestre. 

'Se' -> [('P', 0.43), ('Se', 0.34), ('H', 0.23), ('Pod', 0.0)]
' puede' -> [(' puede', 0.99), (' pueden', 0.0), (' podrá', 0.0), (' pode', 0.0)]
' ins' -> [(' ins', 1.0), (' inscription', 0.0), (' registrar', 0.0), (' registr', 0.0)]
'cri' -> [('cri', 1.0), ('cr', 0.0), ('cre', 0.0), ('cript', 0.0)]
'bir' -> [('bir', 1.0), ('b', 0.0), ('bid', 0.0), ('ber', 0.0)]
' dos' -> [(' dos', 0.48), (' hasta', 0.46), ('se', 0.06), (' durante', 0.0)]


**3. Curación**

se descartan las respuestas muy cortas, las que quedaron cortadas por el límite de tokens y las que traen razonamiento visible

In [ ]:
dataset = [
    g for g in generaciones
    if len(g["texto"]) >= 15 and g["termino"] and "<think>" not in g["texto"]
]
print("generadas:", len(generaciones), " conservadas:", len(dataset))

# el maestro ya no hace falta: se libera la memoria de la gpu
del maestro, salida, logits
gc.collect()
torch.cuda.empty_cache()

generadas: 120  conservadas: 119


**4. La pérdida**

Suma dos términos:

- **Entropía cruzada**: que el alumno prediga el token que eligió el maestro.
- **Divergencia KL**: que la distribución del alumno sobre los 32 candidatos se parezca a la del maestro.

La salida del modelo en la posición t predice el token t+1, por eso todo se corre una
posición. Si se omite ese corrimiento, el código igual corre, pero entrena mal

In [ ]:
def perdida_destilacion(logits, etiquetas, candidatos, probs, mascara):
    # la posicion t predice el token t+1
    logits = logits[:, :-1].float()
    etiquetas = etiquetas[:, 1:]
    candidatos = candidatos[:, 1:]
    probs = probs[:, 1:]
    mascara = mascara[:, 1:]

    ce = F.cross_entropy(logits.reshape(-1, logits.size(-1)), etiquetas.reshape(-1), ignore_index=-100)

    # alumno: sus logits en los candidatos del maestro, suavizados y renormalizados
    log_alumno = F.log_softmax(torch.gather(logits, 2, candidatos) / temperatura, dim=-1)
    # maestro: sus probabilidades suavizadas con la misma temperatura y renormalizadas
    maestro_t = probs.clamp_min(1e-10) ** (1 / temperatura)
    maestro_t = maestro_t / maestro_t.sum(-1, keepdim=True)

    kl = (maestro_t * (maestro_t.log() - log_alumno)).sum(-1)
    kl = (kl * mascara).sum() / mascara.sum().clamp_min(1) * temperatura ** 2

    return peso_ce * ce + peso_kl * kl, ce.item(), kl.item()


# prueba rapida con numeros inventados: si el alumno copia al maestro la kl da cero
c = torch.stack([torch.randperm(50)[:8] for _ in range(5)]).unsqueeze(0)   # 8 candidatos distintos por paso
p = F.softmax(torch.randn(1, 5, 8), dim=-1)
l = torch.full((1, 5, 50), -30.0)
for t in range(1, 5):
    l[0, t - 1, c[0, t]] = p[0, t].log()
_, _, kl_prueba = perdida_destilacion(l, c[:, :, 0], c, p, torch.ones(1, 5))
print("kl cuando el alumno copia al maestro:", abs(round(kl_prueba, 6)))

kl cuando el alumno copia al maestro: 0.0


**5. Preparar los lotes**

* cada ejemplo es el prompt seguido de los tokens que generó el maestro
* se usan esos tokens y no el texto vuelto a tokenizar, porque al retokenizar puede salir otra secuencia y las posiciones dejarían de coincidir con las distribuciones guardadas

In [ ]:
def armar_lote(ejemplos):
    n = len(ejemplos) # cuantos ejemplos entran en el lote
    largo = max(len(e["prompt"]) + len(e["respuesta"]) for e in ejemplos) # largo del ejemplo mas largo, todos se igualan a este
    ids = torch.full((n, largo), tokenizador.pad_token_id) # tabla de tokens, llena de relleno al inicio
    atencion = torch.zeros((n, largo), dtype=torch.long) # 1 donde hay token real, 0 donde hay relleno
    etiquetas = torch.full((n, largo), -100) # token a predecir, -100 significa ignorar esa posicion
    candidatos = torch.zeros((n, largo, top_k), dtype=torch.long) # los 32 tokens candidatos del maestro por posicion
    probs = torch.zeros((n, largo, top_k)) # la probabilidad que el maestro le dio a cada candidato
    mascara = torch.zeros((n, largo)) # 1 donde se calcula la kl, 0 donde no


    "cual es la capital de españa? la capital de españa es madrid"
    ------------------------------ = 10
                                    ------------------------------- = 11
     -100, -100,                    90 , 91092,9 09...
    for i, e in enumerate(ejemplos):
        a = len(e["prompt"]) # posicion donde empieza la respuesta
        b = a + len(e["respuesta"]) # posicion donde termina la respuesta
        ids[i, :b] = torch.tensor(e["prompt"] + e["respuesta"]) # copia prompt + respuesta; lo que sobra queda como relleno
        atencion[i, :b] = 1 # marca como reales los tokens copiados
        etiquetas[i, a:b] = torch.tensor(e["respuesta"]) # solo la respuesta es objetivo; el prompt queda en -100
        candidatos[i, a:b] = e["candidatos"] # pone los candidatos del maestro en las posiciones de la respuesta
        probs[i, a:b] = e["probs"] # y sus probabilidades en las mismas posiciones
        mascara[i, a:b] = 1 # la kl se mide solo sobre la respuesta

    lote_dict = {"ids": ids, "atencion": atencion, "etiquetas": etiquetas,
                 "candidatos": candidatos, "probs": probs, "mascara": mascara} # junta todo en un diccionario
    return {k: v.to(dispositivo) for k, v in lote_dict.items()} # manda cada tabla a la gpu y las devuelve


prueba = armar_lote(dataset[:2]) # arma un lote de prueba con dos ejemplos
print({k: tuple(v.shape) for k, v in prueba.items()}) # muestra la forma de cada tabla para comprobar que cuadran

{'ids': (2, 156), 'atencion': (2, 156), 'etiquetas': (2, 156), 'candidatos': (2, 156, 32), 'probs': (2, 156, 32), 'mascara': (2, 156)}


**6. Alumno**

* DoRA es una variante de LoRA: congela los pesos del modelo y entrena solo un
adaptador chico
* separa cada peso en magnitud y dirección y ajusta las dos por
separado
* pasar de LoRA a DoRA es una sola opción: use_dora=True

Antes de entrenar se mide al alumno con el adaptador apagado, así la comparación usa
exactamente el mismo modelo y los mismos prompts

In [ ]:
def normalizar(texto):
    texto = unicodedata.normalize("NFKD", texto.lower()) #("NFKD", ...) separa cada letra con tilde en dos caracteres, la letra sola más la tilde como una marca aparte.
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return re.sub(r"[^\w\s]", " ", texto).split()


def f1(prediccion, referencia):
    # proporcion de palabras compartidas entre la respuesta y la referencia
    pred = normalizar(prediccion)
    ref = normalizar(referencia)
    comunes = 0
    for palabra in pred:
        if palabra in ref:
            ref.remove(palabra)
            comunes += 1
    if comunes == 0:
        return 0.0
    precision = comunes / len(pred)
    recall = comunes / len(normalizar(referencia))
    return 2 * precision * recall / (precision + recall)


def responder(modelo, items):
    respuestas = []
    for i in range(0, len(items), lote_generacion):
        entrada = tokenizador([construir_prompt(x) for x in items[i:i + lote_generacion]],
                              return_tensors="pt", padding=True, add_special_tokens=False).to(dispositivo)
        with torch.no_grad(), torch.autocast(dispositivo, dtype=tipo, enabled=dispositivo == "cuda"):
            salida = modelo.generate(**entrada, max_new_tokens=max_tokens, do_sample=False,
                                     pad_token_id=tokenizador.pad_token_id)
        for fila in salida[:, entrada["input_ids"].shape[1]:]:
            respuestas.append(tokenizador.decode(fila, skip_special_tokens=True).strip())
    return respuestas


# el alumno se carga en float32 y calcula en media precision con autocast:
# es pequeño y el optimizador trabaja con precision completa
alumno = AutoModelForCausalLM.from_pretrained(modelo_alumno, dtype=torch.float32).to(dispositivo)
alumno = get_peft_model(alumno, LoraConfig(
    r=rango,
    lora_alpha=2 * rango,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
    use_dora=usar_dora,
))
alumno.print_trainable_parameters()

alumno.eval()
with alumno.disable_adapter():
    respuestas_antes = responder(alumno, evaluacion)
f1_antes = [f1(r, x["referencia"]) for r, x in zip(respuestas_antes, evaluacion)]
print("f1 medio antes de destilar:", round(np.mean(f1_antes), 3))

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

trainable params: 10,436,608 || all params: 606,486,528 || trainable%: 1.7208
f1 medio antes de destilar: 0.31


**7. Entrenar**

Un ciclo de entrenamiento común:
* armar el lote
* calcular la pérdida
* retropropagar
* actualizar

la tasa de aprendizaje sube al principio y baja con forma de coseno

In [ ]:
entrenables = [p for p in alumno.parameters() if p.requires_grad] # solo los pesos del adaptador, el resto esta congelado
optimizador = torch.optim.AdamW(entrenables, lr=tasa_aprendizaje) # optimizador que actualiza esos pesos
pasos_totales = epocas * math.ceil(len(dataset) / lote) # cuantas actualizaciones habra en total
calentamiento = max(1, pasos_totales // 10) # el primer 10% de pasos sube la tasa de a poco
planificador = torch.optim.lr_scheduler.LambdaLR( # ajusta la tasa de aprendizaje en cada paso
    optimizador,
    lambda paso: min(1.0, (paso + 1) / calentamiento) * 0.5 * (1 + math.cos(math.pi * paso / pasos_totales)), # sube al inicio y despues baja en forma de coseno hasta 0
)
# con float16 hace falta escalar la perdida para que los gradientes no se anulen
escalador = torch.amp.GradScaler(enabled=tipo == torch.float16) # solo se activa con float16, en la t4

alumno.train() # modo entrenamiento, activa el dropout
for epoca in range(epocas): # una epoca es una pasada por todo el dataset
    random.shuffle(dataset) # mezcla el orden en cada epoca
    suma_ce = 0.0 # acumuladores para el promedio de la epoca
    suma_kl = 0.0
    n_lotes = 0
    for i in range(0, len(dataset), lote): # recorre el dataset de a 4 ejemplos
        b = armar_lote(dataset[i:i + lote]) # arma las tablas del lote
        with torch.autocast(dispositivo, dtype=tipo, enabled=dispositivo == "cuda"): # calcula en media precision, mas rapido
            logits = alumno(input_ids=b["ids"], attention_mask=b["atencion"]).logits # el alumno predice el siguiente token en cada posicion
        perdida, ce, kl = perdida_destilacion(logits, b["etiquetas"], b["candidatos"], b["probs"], b["mascara"]) # compara con el maestro

        optimizador.zero_grad() # borra los gradientes del paso anterior
        escalador.scale(perdida).backward() # calcula los gradientes, con la perdida agrandada
        escalador.unscale_(optimizador) # devuelve los gradientes a su escala real
        torch.nn.utils.clip_grad_norm_(entrenables, 1.0) # recorta gradientes muy grandes para que no desestabilicen
        escalador.step(optimizador) # actualiza los pesos, salvo que haya gradientes invalidos
        escalador.update() # ajusta el factor de escala para el proximo paso
        planificador.step() # avanza la tasa de aprendizaje un paso

        suma_ce += ce # suma la entropia cruzada del lote
        suma_kl += kl # suma la kl del lote
        n_lotes += 1 # cuenta los lotes
    print(f"epoca {epoca + 1}: entropia cruzada {suma_ce / n_lotes:.3f}  kl {suma_kl / n_lotes:.3f}") # promedio de la epoca

epoca 1: entropia cruzada 0.365  kl 0.833
epoca 2: entropia cruzada 0.159  kl 0.358
epoca 3: entropia cruzada 0.106  kl 0.214


**8. Comparar antes y después**

se usan las 20 preguntas que el alumno nunca vio

In [ ]:
alumno.eval()
respuestas_despues = responder(alumno, evaluacion)
f1_despues = [f1(r, x["referencia"]) for r, x in zip(respuestas_despues, evaluacion)]

diferencias = np.array(f1_despues) - np.array(f1_antes)
remuestreo = np.random.default_rng(0).integers(0, len(diferencias), (10000, len(diferencias)))
bajo, alto = np.percentile(diferencias[remuestreo].mean(axis=1), [2.5, 97.5])

print("f1 antes:  ", round(np.mean(f1_antes), 3))
print("f1 despues:", round(np.mean(f1_despues), 3))
print(f"diferencia: {diferencias.mean():+.3f}  intervalo 95%: [{bajo:+.3f}, {alto:+.3f}]")
print("mejora en", (diferencias > 0).sum(), "preguntas, empeora en", (diferencias < 0).sum())

for i in range(0, len(evaluacion), 4):
    print("\npregunta:  ", evaluacion[i]["pregunta"])
    print("referencia:", evaluacion[i]["referencia"])
    print("antes:     ", respuestas_antes[i])
    print("despues:   ", respuestas_despues[i])

f1 antes:   0.31
f1 despues: 0.361
diferencia: +0.052  intervalo 95%: [-0.034, +0.135]
mejora en 11 preguntas, empeora en 8

pregunta:   Me inscribo tarde, ¿todavia llego?
referencia: Se habilita dos semanas antes del cuatrimestre y cierra el viernes de la primera semana de cursada.
antes:      El contexto no proporciona información sobre la inscripción.
despues:    Sí, si te inscribes tarde, aún llega.

pregunta:   Ya curse algo parecido en otra facultad, ¿sirve?
referencia: Una nota a la direccion del posgrado con el programa de la materia y el certificado analitico.
antes:      El contexto no proporciona información sobre si el curso en otra facultad es similar al que se menciona.
despues:    Sí, se puede cursar algo parecido en otra facultad, ya que se permite solicitar equivalencias mediante una nota dirigida a la dirección del posgrado.

pregunta:   ¿Que pasa si dejo de pagar dos meses?
referencia: Se paga en diez cuotas; una mora mayor a sesenta dias suspende la inscripcion a nu



Se guarda solo el adaptador para usarlo después:
PeftModel.from_pretrained(modelo_base, "adaptador").

In [ ]:
alumno.save_pretrained("adaptador")
print("adaptador guardado en la carpeta adaptador")

adaptador guardado en la carpeta adaptador
